# LTFSR-Meta — Run ALL methods in one pass

This notebook trains **every method back-to-back** (no manual `METHOD` switching),
saves each run to `outputs/<method>/`, then draws the comparison plots — including
**one figure per measure with all methods overlaid**.

Methods: `baseline` → `balanced_softmax` → `decoupling` → `supcon` → `meta` (bonus).
Headline metric = **balanced_accuracy** and **few_shot_accuracy**, not plain top-1.

## 1. Make `src/` importable (works locally and on Kaggle)

In [ ]:
import sys
from pathlib import Path


def find_project_root() -> Path:
    """Return the folder containing `src/` (search cwd, parents, Kaggle inputs)."""
    candidates = [Path.cwd(), *Path.cwd().parents]
    for base in (Path("/kaggle/input"), Path("/kaggle/working")):
        if base.exists():
            candidates += [p for p in base.iterdir() if p.is_dir()]
    for path in candidates:
        if (path / "src" / "__init__.py").exists():
            return path
    return Path.cwd()


PROJECT_DIR = find_project_root()
sys.path.insert(0, str(PROJECT_DIR))
print("Project root:", PROJECT_DIR)

## 2. Configuration — the only cell you normally edit

`METHODS` is the list that gets trained, in order. Remove any you don't want.

In [ ]:
# --- paths ---
DATA_DIR = PROJECT_DIR / "data" / "CIFAR-100-LT"   # folder with train/ test/ class_counts.json
OUTPUT_DIR = PROJECT_DIR / "outputs"               # where run folders are written
BUILD_DATASET = False   # set True on Kaggle to build CIFAR-100-LT into OUTPUT_DIR first

# --- which methods to run (in order) ---
METHODS = ["baseline", "balanced_softmax", "decoupling", "supcon", "meta", "cmo"]
# Phase-1 augmentation methods also available: "mixup", "cutmix" (add to the list above).

# --- setup: MAIN (from scratch) vs optional PRETRAINED reference table ---
# PRETRAINED=False -> CIFAR stem, train from scratch on 32x32  (MAIN; keep IMAGE_SIZE=32)
# PRETRAINED=True  -> ImageNet ResNet-18; you MUST set IMAGE_SIZE=224 (optional table only)
PRETRAINED = False
IMAGE_SIZE = 32

# --- core hyperparameters (shared by all methods) ---
NUM_CLASSES = 100
BATCH_SIZE = 128
LEARNING_RATE = 0.1
EPOCHS = 200            # from-scratch CIFAR needs a long schedule for good results
SEED = 42
NUM_WORKERS = 2
DEVICE = None           # None = auto (CUDA if available), or "cpu" / "cuda"

# --- quick smoke test (None = full dataset). Try MAX_TRAIN_SAMPLES=2000, EPOCHS=5 first. ---
MAX_TRAIN_SAMPLES = None
MAX_TEST_SAMPLES = None

# --- validation split for model selection (held out from train; test stays untouched) ---
VAL_FRACTION = 0.1

# --- decoupling / cRT classifier-retraining stage (Methods 3 & 4) ---
CRT_EPOCHS = 10
CRT_LR = 0.1

# --- SupCon contrastive pre-training (Method 4) ---
PRETRAIN_EPOCHS = 200
PRETRAIN_LR = 0.5
TEMPERATURE = 0.07

# --- few-shot / meta-learning (Method 5, bonus) ---
N_WAY = 5
K_SHOT = 5
N_QUERY = 15
EPISODES_PER_EPOCH = 100
META_LR = 0.001

# --- Phase 1: mixing augmentation (methods "mixup" / "cutmix" / "cmo") ---
AUG_ALPHA = 1.0
MIX_PROB = 0.5

## 3. Imports, reproducibility, device

In [ ]:
import random

import numpy as np
import torch

from src.datasets.cifar_lt import (build_transforms, load_class_counts, load_split,
                                    make_loader, split_indices_by_class, split_shot_groups,
                                    subset)
from src.evaluation import visualize
from src.evaluation.metrics import compute_metrics, format_metrics
from src.trainers.engine import evaluate
from src.utils.experiment import (compare_runs, create_run_dir, save_config,
                                   save_history, save_metrics)
from src.utils.seed import resolve_device, set_seed

device = resolve_device(DEVICE)
print("Device:", device)

## 4. (Optional) Build the dataset, then load it once

In [ ]:
if BUILD_DATASET:
    import subprocess
    subprocess.run([sys.executable, str(PROJECT_DIR / "data" / "prepare_datasets.py"),
                    "--dataset", "cifar100-lt",
                    "--data_dir", str(OUTPUT_DIR), "--overwrite"], check=True)
    DATA_DIR = OUTPUT_DIR / "CIFAR-100-LT"

assert (DATA_DIR / "train").exists(), f"No train/ folder under {DATA_DIR}"
class_counts = load_class_counts(DATA_DIR)
shot_groups = split_shot_groups(class_counts)
print("Classes per group:", {name: len(ids) for name, ids in shot_groups.items()})
print("Head class count:", max(class_counts.values()), "| Tail class count:", min(class_counts.values()))

In [ ]:
# Load full train (augmented + eval views) and the test set.
base_aug = load_split(DATA_DIR, "train", build_transforms(train=True, image_size=IMAGE_SIZE))
base_eval = load_split(DATA_DIR, "train", build_transforms(train=False, image_size=IMAGE_SIZE))
test_eval = load_split(DATA_DIR, "test", build_transforms(train=False, image_size=IMAGE_SIZE))

# Optional smoke-test subsample (same indices for both train views, so they stay aligned).
if MAX_TRAIN_SAMPLES is not None and MAX_TRAIN_SAMPLES < len(base_aug.samples):
    keep = sorted(random.Random(SEED).sample(range(len(base_aug.samples)), MAX_TRAIN_SAMPLES))
    base_aug, base_eval = subset(base_aug, keep), subset(base_eval, keep)
if MAX_TEST_SAMPLES is not None and MAX_TEST_SAMPLES < len(test_eval.samples):
    test_keep = sorted(random.Random(SEED).sample(range(len(test_eval.samples)), MAX_TEST_SAMPLES))
    test_eval = subset(test_eval, test_keep)

# Stratified train/val split. val is used ONLY for model selection; test is never touched.
train_idx, val_idx = split_indices_by_class([label for _, label in base_aug.samples], VAL_FRACTION, SEED)
train_aug = subset(base_aug, train_idx)     # training (augmented)
train_eval = subset(base_eval, train_idx)   # prototypes / t-SNE (eval transform)
val_eval = subset(base_eval, val_idx)       # model selection (eval transform)

train_loader = make_loader(train_aug, BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
train_eval_loader = make_loader(train_eval, BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
val_loader = make_loader(val_eval, BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = make_loader(test_eval, BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
print(f"train={len(train_aug)} | val={len(val_eval)} | test={len(test_eval)} images")

# Look at the data once (long-tail profile).
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
visualize.plot_class_frequency(class_counts, OUTPUT_DIR)
visualize.plot_shot_distribution(shot_groups, OUTPUT_DIR)

## 5. Train every method back-to-back

Each method is reseeded for a fair start, trained, evaluated on the test set, and
its config / metrics / history saved to `outputs/<method>/`. Histories are kept in
memory for the overlay plots.

In [ ]:
def train_one(method, run_dir):
    """Dispatch one method. Returns (result_dict, history_df)."""
    if method == "baseline":
        from src.trainers.baseline_trainer import train_baseline
        model, history = train_baseline(train_loader, val_loader, NUM_CLASSES, device, run_dir,
                                        epochs=EPOCHS, learning_rate=LEARNING_RATE, pretrained=PRETRAINED)
        return evaluate(model, test_loader, device, collect_features=True), history

    if method == "balanced_softmax":
        from src.trainers.baseline_trainer import train_baseline
        from src.trainers.losses import BalancedSoftmaxLoss
        counts = [class_counts[c] for c in range(NUM_CLASSES)]
        model, history = train_baseline(train_loader, val_loader, NUM_CLASSES, device, run_dir,
                                        epochs=EPOCHS, learning_rate=LEARNING_RATE, pretrained=PRETRAINED,
                                        criterion=BalancedSoftmaxLoss(counts))
        return evaluate(model, test_loader, device, collect_features=True), history

    if method == "decoupling":
        from src.trainers.decoupling_trainer import train_decoupling
        model, history = train_decoupling(train_loader, train_aug, val_loader, NUM_CLASSES, device, run_dir,
                                          epochs=EPOCHS, learning_rate=LEARNING_RATE,
                                          crt_epochs=CRT_EPOCHS, crt_lr=CRT_LR,
                                          batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pretrained=PRETRAINED)
        return evaluate(model, test_loader, device, collect_features=True), history

    if method == "supcon":
        from src.trainers.contrastive_trainer import train_contrastive
        model, history, _ = train_contrastive(
            train_aug, val_loader, NUM_CLASSES, device, run_dir,
            pretrain_epochs=PRETRAIN_EPOCHS, probe_epochs=CRT_EPOCHS,
            batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
            pretrain_lr=PRETRAIN_LR, probe_lr=CRT_LR,
            temperature=TEMPERATURE, pretrained=PRETRAINED, image_size=IMAGE_SIZE)
        return evaluate(model, test_loader, device, collect_features=True), history

    if method == "meta":
        from src.trainers.meta_trainer import evaluate_meta, train_meta
        encoder, history = train_meta(train_aug, val_eval, device, run_dir,
                                      epochs=EPOCHS, episodes_per_epoch=EPISODES_PER_EPOCH,
                                      n_way=N_WAY, k_shot=K_SHOT, n_query=N_QUERY,
                                      learning_rate=META_LR, pretrained=PRETRAINED, seed=SEED)
        return evaluate_meta(encoder, train_eval_loader, test_loader, NUM_CLASSES, device), history

    if method in ("mixup", "cutmix", "cmo"):
        from src.trainers.augment_trainer import train_augmented
        model, history = train_augmented(train_aug, val_loader, NUM_CLASSES, device, run_dir, class_counts,
                                         mode=method, alpha=AUG_ALPHA, mix_prob=MIX_PROB,
                                         epochs=EPOCHS, learning_rate=LEARNING_RATE,
                                         batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                                         pretrained=PRETRAINED, use_balanced_softmax=True)
        return evaluate(model, test_loader, device, collect_features=True), history

    raise ValueError(f"Unknown method: {method!r}")


histories = {}        # method -> per-epoch history (for overlay curves)
all_results = {}      # method -> result dict (for confusion matrices / t-SNE)

for method in METHODS:
    print(f"\n========== {method} ==========")
    set_seed(SEED)                                   # fair, reproducible start per method
    run_dir = create_run_dir(OUTPUT_DIR, run_name=method)
    result, history = train_one(method, run_dir)

    metrics = compute_metrics(result["y_true"], result["y_pred"], NUM_CLASSES, shot_groups)
    print(format_metrics(metrics))

    save_config(run_dir, {"method": method, "epochs": EPOCHS, "pretrained": PRETRAINED,
                          "image_size": IMAGE_SIZE, "seed": SEED, "batch_size": BATCH_SIZE})
    save_metrics(run_dir, metrics)
    save_history(run_dir, history)
    # Per-method figures (also saved in each run folder).
    visualize.plot_confusion_matrices(result["y_true"], result["y_pred"], NUM_CLASSES, run_dir)
    if "features" in result:
        visualize.plot_tsne(result["features"], result["y_true"], run_dir)

    histories[method] = history
    all_results[method] = result

print("\nAll methods finished.")

## 6. Comparison table (headline table for the report)

One row per method, sorted by balanced accuracy.

In [ ]:
comparison = compare_runs(OUTPUT_DIR)
comparison.to_csv(OUTPUT_DIR / "comparison.csv", index=False)
comparison

## 7. Comparison plots — all methods together

- **Bar chart**: every method side by side across the key metrics (one figure).
- **Overlay curves**: one figure per measure, with all methods drawn as lines.

> `meta` is excluded from the training-curve overlay because its `val_accuracy` is
> an *episodic N-way* score, not a 100-way test score — mixing them would mislead.
> It still appears in the bar chart, where the metrics are computed the same way.

In [ ]:
# 1) Grouped bar chart across metrics (all methods in one figure)
visualize.plot_metric_comparison(comparison, OUTPUT_DIR)

# 2) One overlay figure per training measure (all comparable methods as lines)
curve_histories = {m: h for m, h in histories.items() if m != "meta"}
overlay_paths = visualize.plot_curves_overlay(curve_histories, OUTPUT_DIR)

print("Saved to", OUTPUT_DIR)
print(sorted(p.name for p in OUTPUT_DIR.glob("*.png")))

In [ ]:
# Show the figures inline
from IPython.display import Image, display

for name in ["comparison_metrics.png", "overlay_val_accuracy.png", "overlay_val_loss.png"]:
    path = OUTPUT_DIR / name
    if path.exists():
        print(name)
        display(Image(filename=str(path)))

## 8. Summary

`outputs/` now holds:
- `comparison.csv` + `comparison_metrics.png` — the headline comparison.
- `overlay_*.png` — one figure per measure with all methods overlaid.
- `outputs/<method>/` — per-method `metrics.json`, `metrics.csv`, checkpoint, confusion matrices, t-SNE.

Expect **balanced_accuracy** and **few_shot_accuracy** to rise across
`baseline → balanced_softmax → decoupling → supcon`.